# X-ray diseases classifier

This notebook demonstrates an image classification approach for analyzing chest X-ray images. The project uses the NIH Chest X-ray dataset and focuses on exploring a proof-of-concept method for detecting thoracic findings from medical imaging data.

Rabi Arbi, 2026

![banner](https://d1qpnt4t1p05jk.cloudfront.net/wp-content/uploads/2025/10/27082644/image-2544-1024x683.jpeg)

In [1]:
import collections, lightning, pandas as pd, platform, os, torch, torchvision, warnings

from torch.utils.data import Dataset, DataLoader
from lightning import LightningDataModule, LightningModule, Trainer
from torchvision import transforms
from torchmetrics.classification import MultilabelAccuracy, MultilabelPrecision, MultilabelRecall
from lightning.pytorch.loggers import CSVLogger
from lightning.pytorch.callbacks import EarlyStopping

import matplotlib.pyplot as plt

lightning.seed_everything(42)
torch.set_float32_matmul_precision("high")

data_path = "data"

print(
    "Versions: python", platform.python_version(),
    "| torch", torch.__version__,
    "| torchvision", torchvision.__version__,
    "| lightning", lightning.__version__
)

Seed set to 42


Versions: python 3.14.0 | torch 2.11.0+cpu | torchvision 0.26.0+cpu | lightning 2.6.1


### explanation

- `collections` → counts labels and checks class balance  
- `lightning` → organizes training structure  
- `pandas` → reads and processes the dataset CSV  
- `platform` → shows Python/version info  
- `os` → handles file paths  
- `torch` → core deep learning library  
- `torchvision` → image utilities and transforms  
- `warnings` → controls warning messages  

- `Dataset` → builds a custom NIH dataset class  
- `DataLoader` → loads images in batches  

- `LightningDataModule` → manages data loading and splits  
- `LightningModule` → defines model, loss, and training steps  
- `Trainer` → runs training and validation  

- `transforms` → preprocesses X-ray images  
- `MultilabelAccuracy` → evaluates multi-label predictions  
- `MultilabelPrecision` → checks how many positive predictions are correct  
- `MultilabelRecall` → checks how many real positives are found  

- `CSVLogger` → saves training results  
- `EarlyStopping` → stops training when improvement stalls  
- `matplotlib.pyplot` → plots images and training curves  

- `seed_everything(42)` → makes runs reproducible  
- `set_float32_matmul_precision("high")` → improves GPU computation performance  
- `data_path = "data"` → sets dataset folder  
- `print(...)` → shows library versions

# Loading a pre-trained model

In [6]:
import torch.nn as nn
import torchvision

num_classes = 14

pretrained_model = torchvision.models.resnet50(weights="DEFAULT")
pretrained_model.fc = nn.Linear(pretrained_model.fc.in_features, num_classes)

criterion = nn.BCEWithLogitsLoss()

## What each part does:

- `resnet50(weights="DEFAULT")` → loads pretrained ResNet50
- `nn.Linear(...)` → replaces the last layer
- `num_classes` = 14 → one output per chest finding
- `BCEWithLogitsLoss()` → correct loss for multi-label prediction

### Why the last layer was replaced

The old last layer was built for a different task.

This pretrained `ResNet50` came from ImageNet, so its final layer was made to predict **1000 ImageNet classes** such as dogs, cars, chairs, and other everyday objects.

This project is different:
- **input** = chest X-ray image  
- **output** = 14 thoracic findings  

So the original last layer does not match this target.

Inside the model:
- the early and middle layers learn general visual features  
- the last layer maps those features to the final class outputs  

That last mapping is task-specific.

The last layer was replaced because:
- the ImageNet output size is wrong  
- this project needs **14 outputs**  
- the new classes are completely different  
- this is how transfer learning adapts a pretrained model to a new task  

Simple example:

**Before**
- model sees an image  
- final layer predicts one of **1000 ImageNet classes**  

**After**
- model sees a chest X-ray  
- final layer predicts the 14 chest findings, such as **Atelectasis, Effusion, Mass, Pneumonia**, and others.

## Prepare the dataset